In [ ]:
"""
🔥 EDA 통합 파이프라인
데이터 로드 → 미확인 값 처리 → Gemini 브랜드 분석 → 지리 피처 생성

실행 순서:
1. 거래 데이터 로드 및 탐색
2. 미확인 값 계층적 처리
3. 카테고리 데이터 전처리
4. Gemini API로 브랜드 메타데이터 추출
5. 카카오맵 API로 지점 위치 및 지리 피처 생성
"""

import pandas as pd
import numpy as np
import re
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

print("=" * 80)
print("🔥 EDA 통합 파이프라인 시작")
print("=" * 80)

# ========================================
# STEP 1: 데이터 로드
# ========================================
print("\n📂 1단계: 데이터 로드")
print("-" * 80)

data = pd.read_csv(r'C:\SEMIN\Project\AUTOML_PIPELINE\train_transactions.csv', encoding='cp949')
print(f"   거래 데이터: {data.shape}")
print(f"   컬럼: {data.columns.tolist()}")
print("\n   샘플 데이터:")
print(data.head())

# Object 타입 컬럼 추출
object_cols = data.select_dtypes(include=['object']).columns.tolist()
goodcd_col = 'goodcd'
object_data = data[object_cols + [goodcd_col]].copy()

print(f"\n   Object 컬럼: {len(object_cols)}개")
print(f"   추출된 데이터: {object_data.shape}")

# 미확인 값 있는 컬럼 확인
print("\n   미확인 값 포함 컬럼:")
unconfirmed_cols = []
for col in object_data.columns:
    if object_data[col].astype(str).str.contains("미확인").any():
        count = object_data[col].astype(str).str.contains("미확인").sum()
        print(f"      • {col}: {count:,}개")
        unconfirmed_cols.append(col)

# ========================================
# STEP 2: 미확인 값 처리
# ========================================
print("\n🔧 2단계: 계층적 미확인 값 처리")
print("-" * 80)

def hierarchical_fill_unconfirmed_detailed(df):
    """계층적으로 미확인 값을 채우기"""
    
    target_cols = ["corner_nm", "pc_nm"]
    hierarchy = ["goodcd", "corner_nm", "pc_nm", "part_nm", "team_nm"]
    
    def is_unconfirmed(series):
        return series.astype(str).str.contains("미확인", regex=True, case=False, na=False)
    
    print("\n   [초기 상태]")
    for col in target_cols:
        if col in df.columns:
            total = len(df)
            unconfirmed = is_unconfirmed(df[col]).sum()
            print(f"      {col:15s}: {unconfirmed:,}개 / {total:,}개 ({unconfirmed/total*100:.2f}%)")
    
    has_unconfirmed = False
    for col in target_cols:
        if col in df.columns and is_unconfirmed(df[col]).sum() > 0:
            has_unconfirmed = True
            break
    
    if not has_unconfirmed:
        print("\n   ⚠️  미확인 값이 없습니다!")
        return df
    
    for target_col in target_cols:
        if target_col not in df.columns:
            continue
        
        initial_unconfirmed = is_unconfirmed(df[target_col]).sum()
        
        if initial_unconfirmed == 0:
            print(f"\n   [{target_col}] 미확인 값이 없어서 스킵")
            continue
        
        print(f"\n   처리 대상: {target_col}")
        
        for level, group_col in enumerate(hierarchy, 1):
            if group_col == target_col:
                continue
            
            before_count = is_unconfirmed(df[target_col]).sum()
            
            if before_count == 0:
                print(f"      ✅ [{group_col}] 모든 미확인 값 채움!")
                break
            
            mask = is_unconfirmed(df[target_col])
            valid_data = df[~mask].copy()
            
            if len(valid_data) == 0:
                continue
            
            mode_dict = (
                valid_data.groupby(group_col)[target_col]
                .agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else None)
                .to_dict()
            )
            
            filled_values = df.loc[mask, group_col].map(mode_dict)
            valid_fills = filled_values.notna()
            df.loc[mask & valid_fills, target_col] = filled_values[valid_fills]
            
            after_count = is_unconfirmed(df[target_col]).sum()
            filled_count = before_count - after_count
            
            if filled_count > 0:
                print(f"      [{group_col}] {filled_count:,}개 채움")
        
        final_unconfirmed = is_unconfirmed(df[target_col]).sum()
        total_filled = initial_unconfirmed - final_unconfirmed
        
        print(f"      → [{target_col}] 총 {total_filled:,}개 채움 ({total_filled/initial_unconfirmed*100:.1f}%)")
    
    print("\n   [최종 미확인 현황]")
    for col in target_cols:
        if col in df.columns:
            unconfirmed = is_unconfirmed(df[col]).sum()
            print(f"      {col:15s}: {unconfirmed:,}개")
    
    return df

# 미확인 값 처리 실행
object_data = hierarchical_fill_unconfirmed_detailed(object_data)

# ========================================
# STEP 3: 카테고리 데이터 전처리
# ========================================
print("\n📊 3단계: 카테고리 데이터 전처리")
print("-" * 80)

try:
    categorical_columns = pd.read_csv(r'C:\SEMIN\Project\AUTOML_PIPELINE\categorical_columns.csv', encoding='cp949')
    print(f"   카테고리 컬럼 데이터: {categorical_columns.shape}")
    
    # 중복 제거
    before_dup = len(categorical_columns)
    categorical_columns = categorical_columns.drop_duplicates()
    after_dup = len(categorical_columns)
    
    print(f"   중복 제거: {before_dup:,}개 → {after_dup:,}개 ({before_dup - after_dup:,}개 제거)")
    print(f"   컬럼명: {categorical_columns.columns.tolist()}")
    
except FileNotFoundError:
    print("   ⚠️  categorical_columns.csv 파일이 없습니다. 스킵합니다.")
    categorical_columns = None

# ========================================
# STEP 4: Gemini API 브랜드 분석
# ========================================
print("\n🤖 4단계: Gemini API 브랜드 메타데이터 추출")
print("-" * 80)

# API 키 설정 (사용자가 입력해야 함)
GEMINI_API_KEY = "HIDDEN"

try:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    
    CACHE_DIR = Path("gemini_cache")
    CACHE_DIR.mkdir(exist_ok=True)
    
    BRAND_CACHE_FILE = CACHE_DIR / "brand_features.json"
    
    def load_cache(filepath):
        if filepath.exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}
    
    def save_cache(cache, filepath):
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    
    def analyze_brand_with_gemini(brand_name, cache):
        """브랜드 → 나이/가격대/스타일 등 메타데이터"""
        
        if brand_name in cache:
            return cache[brand_name]
        
        try:
            model = genai.GenerativeModel('gemini-2.0-flash-exp')
            
            prompt = f"""
백화점 브랜드 "{brand_name}"를 분석해주세요.

JSON 형식으로만 반환:

{{
  "category": "메인 카테고리",
  "sub_category": "세부 카테고리",
  "target_age_min": 주요 타겟 최소 나이,
  "target_age_max": 주요 타겟 최대 나이,
  "target_age_mean": 주요 타겟 평균 나이,
  "price_level": 가격대 (1~5),
  "style": "스타일 키워드",
  "gender_target": "주요 성별",
  "brand_tier": "등급",
  "gift_ratio": 선물용 구매 비율,
  "formality": 격식도 (1~5)
}}
"""
            
            response = model.generate_content(prompt)
            result_text = response.text.strip()
            
            if '```json' in result_text:
                result_text = result_text.split('```json')[1].split('```')[0].strip()
            elif '```' in result_text:
                result_text = result_text.split('```')[1].split('```')[0].strip()
            
            result = json.loads(result_text)
            
            cache[brand_name] = result
            save_cache(cache, BRAND_CACHE_FILE)
            
            time.sleep(0.5)
            
            return result
            
        except Exception as e:
            return {
                "category": "기타", "sub_category": "기타",
                "target_age_min": 30, "target_age_max": 50, "target_age_mean": 40,
                "price_level": 3, "style": "캐주얼", "gender_target": "유니섹스",
                "brand_tier": "일반", "gift_ratio": 0.2, "formality": 3
            }
    
    def create_brand_features(df):
        """브랜드 피처 생성"""
        
        print("   브랜드 메타데이터 추출 중...")
        
        cache = load_cache(BRAND_CACHE_FILE)
        print(f"   기존 캐시: {len(cache)}개")
        
        if 'brd_nm' not in df.columns:
            print("   ⚠️  'brd_nm' 컬럼이 없습니다.")
            return df
        
        unique_brands = df['brd_nm'].dropna().unique()
        print(f"   처리할 브랜드: {len(unique_brands)}개")
        
        brand_analysis = {}
        
        for i, brand in enumerate(unique_brands, 1):
            if i % 10 == 0:
                print(f"      진행: {i}/{len(unique_brands)}")
            brand_analysis[brand] = analyze_brand_with_gemini(brand, cache)
        
        # 피처 매핑
        print("\n   브랜드 피처 생성 중...")
        df['brand_category'] = df['brd_nm'].map({k: v['category'] for k, v in brand_analysis.items()})
        df['brand_sub_category'] = df['brd_nm'].map({k: v['sub_category'] for k, v in brand_analysis.items()})
        df['brand_target_age_min'] = df['brd_nm'].map({k: v['target_age_min'] for k, v in brand_analysis.items()})
        df['brand_target_age_max'] = df['brd_nm'].map({k: v['target_age_max'] for k, v in brand_analysis.items()})
        df['brand_target_age_mean'] = df['brd_nm'].map({k: v['target_age_mean'] for k, v in brand_analysis.items()})
        df['brand_price_level'] = df['brd_nm'].map({k: v['price_level'] for k, v in brand_analysis.items()})
        df['brand_style'] = df['brd_nm'].map({k: v['style'] for k, v in brand_analysis.items()})
        df['brand_gender_target'] = df['brd_nm'].map({k: v['gender_target'] for k, v in brand_analysis.items()})
        df['brand_tier'] = df['brd_nm'].map({k: v['brand_tier'] for k, v in brand_analysis.items()})
        df['brand_gift_ratio'] = df['brd_nm'].map({k: v['gift_ratio'] for k, v in brand_analysis.items()})
        df['brand_formality'] = df['brd_nm'].map({k: v['formality'] for k, v in brand_analysis.items()})
        
        print(f"   ✅ 브랜드 피처 11개 생성 완료!")
        
        return df
    
    # 브랜드 피처 생성 실행
    if categorical_columns is not None:
        categorical_columns = create_brand_features(categorical_columns)
        
        # 저장
        categorical_columns.to_csv('enriched_categorical_validated.csv', index=False, encoding='utf-8-sig')
        print(f"\n   💾 저장: enriched_categorical_validated.csv")
    
except ImportError:
    print("   ⚠️  google-generativeai 패키지가 없습니다. 브랜드 분석을 스킵합니다.")
    print("   설치: pip install google-generativeai")
except Exception as e:
    print(f"   ⚠️  브랜드 분석 중 오류: {e}")

# ========================================
# STEP 5: 지리 피처 생성
# ========================================
print("\n🗺️  5단계: 지리 피처 생성 (카카오맵 API)")
print("-" * 80)

# 카카오맵 API 키 (사용자가 입력해야 함)
MY_REST_API_KEY = "HIDDEN"

try:
    import requests
    from haversine import haversine
    
    def get_hyundai_dept_coordinates(api_key):
        url = "https://dapi.kakao.com/v2/local/search/keyword.json"
        headers = {"Authorization": f"KakaoAK {api_key}"}
        params = {"query": "현대백화점", "page": 1, "size": 15}
        
        places = []
        while True:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code != 200:
                break
                
            data = response.json()
            documents = data.get('documents')
            if not documents:
                break
                
            for place in documents:
                places.append({
                    "지점명": place.get("place_name"),
                    "주소": place.get("road_address_name"),
                    "위도(lat)": place.get("y"),
                    "경도(lon)": place.get("x")
                })
                
            if data.get('meta', {}).get('is_end'):
                break
            params['page'] += 1
        
        return pd.DataFrame(places)
    
    print("   현대백화점 위치 데이터 수집 중...")
    geo_df = get_hyundai_dept_coordinates(MY_REST_API_KEY)
    
    # 타겟 지점 필터링
    target_stores = ['무역점', '본점', '천호점', '신촌점']
    geo_df = geo_df[geo_df['지점명'].str.contains('|'.join(target_stores), na=False)]
    
    print(f"   타겟 지점: {len(geo_df)}개")
    print(geo_df[['지점명', '주소']])
    
    # 랜드마크 정의
    LANDMARKS_SAFE_2000 = {
        "명동": (37.5636, 126.9826),
        "종로": (37.5703, 126.9911),
        "강남역": (37.4980, 127.0276),
        "압구정": (37.5274, 127.0396),
        "잠실": (37.5111, 127.0980),
        "여의도": (37.5219, 126.9245),
        "신촌": (37.5559, 126.9364),
        "한강_여의도": (37.5239, 126.9189),
    }
    
    print("\n   랜드마크 거리 피처 생성 중...")
    for name, coord in LANDMARKS_SAFE_2000.items():
        geo_df[f'dist_{name}'] = geo_df.apply(
            lambda row: haversine(
                (float(row['위도(lat)']), float(row['경도(lon)'])),
                coord,
                unit='km'
            ),
            axis=1
        )
    
    # 지하철역
    SUBWAY_BEFORE_2000 = {
        "강남역": (37.4980, 127.0276),
        "신촌역": (37.5559, 126.9364),
        "천호역": (37.5386, 127.1236),
        "잠실역": (37.5133, 127.1000),
    }
    
    print("   지하철 접근성 피처 생성 중...")
    for name, coord in SUBWAY_BEFORE_2000.items():
        geo_df[f'subway_{name}'] = geo_df.apply(
            lambda row: haversine(
                (float(row['위도(lat)']), float(row['경도(lon)'])),
                coord,
                unit='km'
            ),
            axis=1
        )
    
    # 파생 피처
    print("   파생 피처 생성 중...")
    distance_cols = [col for col in geo_df.columns if col.startswith('dist_')]
    subway_cols = [col for col in geo_df.columns if col.startswith('subway_')]
    
    geo_df['dist_nearest_commercial'] = geo_df[['dist_명동', 'dist_강남역', 'dist_여의도', 'dist_잠실']].min(axis=1)
    geo_df['dist_nearest_subway'] = geo_df[subway_cols].min(axis=1)
    geo_df['landmark_count_3km'] = (geo_df[distance_cols] <= 3).sum(axis=1)
    geo_df['subway_count_1km'] = (geo_df[subway_cols] <= 1).sum(axis=1)
    
    # 복합 지수
    print("   복합 지수 계산 중...")
    geo_df['accessibility_score'] = (
        (1 / (1 + geo_df['dist_nearest_commercial'])) * 0.4 +
        (1 / (1 + geo_df['dist_nearest_subway'])) * 0.4 +
        (geo_df['landmark_count_3km'] / 8) * 0.2
    )
    
    # 저장
    geo_df.to_csv('hyundai_dept_features.csv', index=False, encoding='utf-8-sig')
    print(f"\n   💾 저장: hyundai_dept_features.csv")
    
    print("\n   [지점별 접근성 점수]")
    for idx, row in geo_df.iterrows():
        print(f"      • {row['지점명']:15s}: {row['accessibility_score']:.3f}")
    
except ImportError:
    print("   ⚠️  haversine 패키지가 없습니다. 지리 피처를 스킵합니다.")
    print("   설치: pip install haversine")
except Exception as e:
    print(f"   ⚠️  지리 피처 생성 중 오류: {e}")

# ========================================
# 최종 결과
# ========================================
print("\n" + "=" * 80)
print("📊 최종 결과 요약")
print("=" * 80)

print(f"\n✅ 처리된 데이터:")
print(f"   - 거래 데이터: {data.shape}")
print(f"   - 미확인 처리 데이터: {object_data.shape}")
if categorical_columns is not None:
    print(f"   - 카테고리 데이터: {categorical_columns.shape}")

print(f"\n💾 생성된 파일:")
if categorical_columns is not None:
    print(f"   - enriched_categorical_validated.csv")
try:
    if 'geo_df' in locals():
        print(f"   - hyundai_dept_features.csv")
except:
    pass

print("\n" + "=" * 80)
print("✅ EDA 파이프라인 완료!")
print("=" * 80)

In [ ]:
"""
🏆 AGE PREDICTION - ADAPTIVE SIGMA v2.6
데이터에 맞는 최적 Sigma 자동 탐색!

전략:
- 브랜드/카테고리 데이터 빈도 분석
- 희소 카테고리 = 높은 sigma
- 풍부한 카테고리 = 낮은 sigma
- 실행 시간: ~5분 (v2.5와 동일)
"""

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
import category_encoders as ce
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import entropy

warnings.filterwarnings("ignore")

SEED_ENCODING = 42
SEED_MODEL = 123

# =========================================================
# 캐시 시스템
# =========================================================
CACHE_DIR = Path("competition_cache")
CACHE_DIR.mkdir(exist_ok=True)


def save_cache(obj, name):
    with open(CACHE_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


def load_cache(name):
    cache_file = CACHE_DIR / f"{name}.pkl"
    if cache_file.exists():
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    return None


# =========================================================
# 🔥 최적 Sigma 계산
# =========================================================


def calculate_optimal_sigma(df, column, min_sigma=1, max_sigma=100):
    """
    데이터 분포에 따라 최적 sigma 계산

    규칙:
    - 카테고리가 많고 희소하면 → 높은 sigma (정보 손실 감수)
    - 카테고리가 적고 풍부하면 → 낮은 sigma (정보 활용)
    """
    value_counts = df[column].value_counts()

    # 통계
    n_categories = len(value_counts)
    median_count = value_counts.median()
    mean_count = value_counts.mean()
    min_count = value_counts.min()

    print(f"\n   📊 {column} 분포 분석:")
    print(f"      카테고리 수: {n_categories}")
    print(f"      중앙값 샘플 수: {median_count:.1f}")
    print(f"      평균 샘플 수: {mean_count:.1f}")
    print(f"      최소 샘플 수: {min_count}")

    # 희소도 점수 (0~1)
    # 1에 가까울수록 희소함
    sparsity_score = 0

    # 1. 카테고리가 많을수록 희소
    if n_categories > 1000:
        sparsity_score += 0.4
    elif n_categories > 500:
        sparsity_score += 0.3
    elif n_categories > 100:
        sparsity_score += 0.2
    else:
        sparsity_score += 0.1

    # 2. 중앙값이 작을수록 희소
    if median_count < 3:
        sparsity_score += 0.3
    elif median_count < 10:
        sparsity_score += 0.2
    elif median_count < 30:
        sparsity_score += 0.1

    # 3. 최소값이 1이면 희소
    if min_count == 1:
        sparsity_score += 0.3

    # Sigma 계산
    # 희소할수록 높은 sigma
    optimal_sigma = min_sigma + (max_sigma - min_sigma) * sparsity_score

    print(f"      희소도 점수: {sparsity_score:.2f}")
    print(f"      → 최적 sigma: {optimal_sigma:.1f}")

    return optimal_sigma


# =========================================================
# 적응형 Target Encoding
# =========================================================


def adaptive_target_encoding(train_df, test_df, target_col, categorical_cols):
    """
    각 컬럼마다 최적 sigma를 자동 계산해서 인코딩
    """
    print(f"\n   🎯 Adaptive Target Encoding...")

    encoded_train = train_df[["custid"]].copy()
    encoded_test = test_df[["custid"]].copy()

    for col in categorical_cols:
        # 최적 sigma 계산
        optimal_sigma = calculate_optimal_sigma(
            train_df, col, min_sigma=1, max_sigma=100
        )

        # 해당 sigma로 인코딩
        encoder = ce.CatBoostEncoder(
            cols=[col], sigma=optimal_sigma, random_state=SEED_ENCODING
        )

        encoded_col_train = encoder.fit_transform(train_df[[col]], train_df[target_col])
        encoded_col_test = encoder.transform(test_df[[col]])

        encoded_train[f"TE_{col}"] = encoded_col_train[col].values
        encoded_test[f"TE_{col}"] = encoded_col_test[col].values

    return encoded_train, encoded_test


# =========================================================
# 빠른 검증
# =========================================================


def quick_submit(X_train, X_test, feature_name, n_folds=3):
    """LightGBM 빠른 검증"""
    print(f"\n   ⚡ Quick Validation: {feature_name}")

    feature_cols = [c for c in X_train.columns if c not in ["custid", "age"]]
    X = X_train[feature_cols].fillna(0)
    y = X_train["age"]
    X_test_features = X_test[feature_cols].fillna(0)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED_MODEL)
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
        model = LGBMRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            num_leaves=31,
            random_state=SEED_MODEL,
            verbosity=-1,
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[val_idx] = model.predict(X.iloc[val_idx])
        test_preds += model.predict(X_test_features) / n_folds

    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"      → CV RMSE: {rmse:.4f}")

    submission = pd.DataFrame({"custid": X_test["custid"], "age": test_preds})
    return submission, rmse


# =========================================================
# 데이터 로더
# =========================================================


def load_data():
    """모든 데이터 로드"""
    print("📂 Loading Data...")

    encodings = ["cp949", "utf-8", "euc-kr"]

    for enc in encodings:
        try:
            trans_train = pd.read_csv("train_transactions.csv", encoding=enc)
            print(f"   ✅ train_transactions.csv ({enc})")
            break
        except:
            continue

    for enc in encodings:
        try:
            trans_test = pd.read_csv("test_transactions.csv", encoding=enc)
            print(f"   ✅ test_transactions.csv ({enc})")
            break
        except:
            continue

    for enc in encodings:
        try:
            y_train = pd.read_csv("y_train.csv", encoding=enc)
            print(f"   ✅ y_train.csv ({enc})")
            break
        except:
            continue

    try:
        brand_meta = pd.read_csv("enriched_categorical_validated.csv", encoding="utf-8")
        print(f"   ✅ enriched_categorical_validated.csv")
    except:
        brand_meta = None
        print(f"   ⚠️ enriched_categorical_validated.csv not found")

    return trans_train, trans_test, y_train, brand_meta


# =========================================================
# Base Features
# =========================================================


def create_base_features(trans_df):
    """기본 거래 피처"""
    print("\n   🔧 Creating base features...")

    df = trans_df.copy()

    df["sales_date"] = pd.to_datetime(
        df["sales_month"].astype(str) + df["sales_day"].astype(str).str.zfill(2),
        format="%m%d",
        errors="coerce",
    )

    df["sales_time_str"] = df["sales_time"].astype(str).str.zfill(4)
    df["hour"] = df["sales_time_str"].str[:2].astype(int)

    def get_timeband(h):
        if h < 12:
            return "morning"
        elif h < 14:
            return "lunch"
        elif h < 18:
            return "afternoon"
        else:
            return "evening"

    df["timeband"] = df["hour"].apply(get_timeband)
    df["is_weekend"] = (df["sales_dayofweek"].isin([6, 7])).astype(int)
    df["is_vacation"] = (df["sales_month"].isin([1, 2, 7, 8])).astype(int)

    df["tot_amt_log"] = np.sign(df["tot_amt"]) * np.log1p(np.abs(df["tot_amt"]))
    df["discount_rate"] = np.where(
        df["tot_amt"] == 0, 0, df["dis_amt"] / (np.abs(df["tot_amt"]) + 1)
    )
    df["is_installment"] = (df["inst_mon"] > 0).astype(int)
    df["is_import"] = (df["import_flg"].fillna(0) == 1).astype(int)

    return df


def calculate_entropy_features(df):
    """Shannon Entropy"""
    print("   🎲 Calculating entropy features...")

    entropy_features = []

    for custid, group in df.groupby("custid"):
        features = {"custid": custid}

        if "brd_nm" in group.columns:
            brand_counts = group["brd_nm"].value_counts(normalize=True)
            features["brand_entropy"] = entropy(brand_counts)
            features["brand_unique_ratio"] = len(brand_counts) / len(group)

        if "brand_category" in group.columns:
            cat_counts = group["brand_category"].value_counts(normalize=True)
            features["category_entropy"] = entropy(cat_counts)

        if "timeband" in group.columns:
            time_counts = group["timeband"].value_counts(normalize=True)
            features["timeband_entropy"] = entropy(time_counts)

        if "str_nm" in group.columns:
            store_counts = group["str_nm"].value_counts(normalize=True)
            features["store_entropy"] = entropy(store_counts)

        amt_bins = pd.cut(group["tot_amt"], bins=3, labels=["low", "mid", "high"])
        amt_counts = amt_bins.value_counts(normalize=True)
        features["amount_entropy"] = entropy(amt_counts)

        entropy_features.append(features)

    return pd.DataFrame(entropy_features).fillna(0)


# =========================================================
# 방법 1: Gemini + Adaptive Sigma
# =========================================================


def method1_gemini_adaptive(train_base, test_base, y_train, brand_meta):
    """방법 1: Gemini + Adaptive Sigma TE"""
    print(f"\n{'🔥'*40}")
    print(f"METHOD 1: Gemini + Adaptive Sigma (SMART!)")
    print(f"{'🔥'*40}")

    cached_data = load_cache("method1_adaptive")
    if cached_data:
        print("   ⚡ Loading from cache...")
        return cached_data["X_train"], cached_data["X_test"]

    if brand_meta is None:
        print("   ⚠️ Brand metadata not found")
        return None, None

    # 브랜드 lookup
    brand_lookup = brand_meta.groupby("brd_nm").first().reset_index()
    keep_cols = [
        "brd_nm",
        "brand_target_age_mean",
        "brand_category",
        "brand_price_level",
        "brand_formality",
    ]
    brand_lookup = brand_lookup[keep_cols]

    # Merge
    train_merged = train_base.merge(brand_lookup, on="brd_nm", how="left")
    test_merged = test_base.merge(brand_lookup, on="brd_nm", how="left")

    for col in ["brand_target_age_mean", "brand_price_level", "brand_formality"]:
        if col in train_merged.columns:
            median_val = train_merged[col].median()
            train_merged[col] = train_merged[col].fillna(median_val)
            test_merged[col] = test_merged[col].fillna(median_val)

    # Age
    age_map = y_train.set_index("custid")["age"]
    train_merged["age"] = train_merged["custid"].map(age_map)
    test_merged["age"] = 0

    # ✅ Adaptive Target Encoding
    te_cols = ["brd_nm", "brand_category", "str_nm"]

    encoded_train, encoded_test = adaptive_target_encoding(
        train_merged, test_merged, "age", te_cols
    )

    # Entropy
    entropy_train = calculate_entropy_features(train_merged)
    entropy_test = calculate_entropy_features(test_merged)

    # 집계
    print("   📊 Aggregating...")

    brand_agg = train_merged.groupby("custid").agg(
        {
            "brand_target_age_mean": ["mean", "std"],
            "brand_price_level": "mean",
            "brand_formality": "mean",
        }
    )
    brand_agg.columns = [
        "_".join(map(str, c)) if isinstance(c, tuple) else c for c in brand_agg.columns
    ]
    brand_agg = brand_agg.reset_index()

    brand_agg_test = test_merged.groupby("custid").agg(
        {
            "brand_target_age_mean": ["mean", "std"],
            "brand_price_level": "mean",
            "brand_formality": "mean",
        }
    )
    brand_agg_test.columns = [
        "_".join(map(str, c)) if isinstance(c, tuple) else c
        for c in brand_agg_test.columns
    ]
    brand_agg_test = brand_agg_test.reset_index()

    trans_agg = train_merged.groupby("custid").agg(
        {
            "tot_amt": ["sum", "mean", "std"],
            "discount_rate": "mean",
            "hour": "mean",
        }
    )
    trans_agg.columns = ["_".join(map(str, c)) for c in trans_agg.columns]
    trans_agg = trans_agg.reset_index()

    trans_agg_test = test_merged.groupby("custid").agg(
        {
            "tot_amt": ["sum", "mean", "std"],
            "discount_rate": "mean",
            "hour": "mean",
        }
    )
    trans_agg_test.columns = ["_".join(map(str, c)) for c in trans_agg_test.columns]
    trans_agg_test = trans_agg_test.reset_index()

    # TE 집계
    te_agg = encoded_train.groupby("custid").agg(["mean", "std"])
    te_agg.columns = [f"{c[0]}_{c[1]}" for c in te_agg.columns]
    te_agg = te_agg.reset_index()

    te_agg_test = encoded_test.groupby("custid").agg(["mean", "std"])
    te_agg_test.columns = [f"{c[0]}_{c[1]}" for c in te_agg_test.columns]
    te_agg_test = te_agg_test.reset_index()

    # 통합
    X_train = (
        brand_agg.merge(trans_agg, on="custid", how="left")
        .merge(te_agg, on="custid", how="left")
        .merge(entropy_train, on="custid", how="left")
        .merge(y_train[["custid", "age"]], on="custid", how="left")
        .fillna(0)
    )

    X_test = (
        brand_agg_test.merge(trans_agg_test, on="custid", how="left")
        .merge(te_agg_test, on="custid", how="left")
        .merge(entropy_test, on="custid", how="left")
        .fillna(0)
    )

    print(f"   ✅ Train: {X_train.shape}, Test: {X_test.shape}")

    save_cache({"X_train": X_train, "X_test": X_test}, "method1_adaptive")
    return X_train, X_test


# =========================================================
# MAIN
# =========================================================


def main():
    """실행"""
    print(f"\n{'🏆'*40}")
    print(f"AGE PREDICTION - ADAPTIVE SIGMA v2.6")
    print(f"데이터 기반 최적 Sigma 자동 탐색!")
    print(f"{'🏆'*40}\n")

    trans_train, trans_test, y_train, brand_meta = load_data()

    # Base Features
    print(f"\n{'⚡'*40}")
    print(f"Creating Base Features")
    print(f"{'⚡'*40}")

    cached_base = load_cache("base_features")
    if cached_base:
        print("   ⚡ Loading from cache...")
        train_base = cached_base["train"]
        test_base = cached_base["test"]
    else:
        train_base = create_base_features(trans_train)
        test_base = create_base_features(trans_test)
        save_cache({"train": train_base, "test": test_base}, "base_features")

    results = []

    # 방법 1
    X_train, X_test = method1_gemini_adaptive(
        train_base, test_base, y_train, brand_meta
    )
    if X_train is not None:
        X_train.to_csv("feature_train_adaptive.csv", index=False, encoding="utf-8-sig")
        X_test.to_csv("feature_test_adaptive.csv", index=False, encoding="utf-8-sig")
        submission, rmse = quick_submit(X_train, X_test, "Adaptive")
        submission.to_csv("submission_adaptive.csv", index=False, encoding="utf-8-sig")
        results.append(("Adaptive_Sigma", rmse))
        print(f"\n   💾 Saved: adaptive files")

    # 결과
    print(f"\n{'='*80}")
    print(f"📊 RESULTS (Adaptive Sigma)")
    print(f"{'='*80}")

    for method, rmse in sorted(results, key=lambda x: x[1]):
        print(f"   {method:25s}: CV RMSE = {rmse:.4f}")

    print(f"\n💡 각 컬럼마다 최적 sigma 자동 계산!")
    print(f"   → 희소한 컬럼 = 높은 sigma (과적합 방지)")
    print(f"   → 풍부한 컬럼 = 낮은 sigma (정보 활용)")
    print(f"{'🎉'*40}\n")


if __name__ == "__main__":
    main()